# Notebook 1: SQL in R
## NorthStar Urban Mobility and Logistics
### Databases and Analytics — University of West London

This notebook executes SQL queries within R using the `sqldf` package, accessed via `rpy2` in Google Colab (Python 3 runtime).

**Learning Outcome:** LO1 — Apply SQL in R analytics for writing efficient database queries.

## Step 1: Upload and Extract Dataset

In [1]:
# Upload the northstar zip file
from google.colab import files
uploaded = files.upload()  # select northstar_dataset__1_.zip

Saving northstar_dataset (1).zip to northstar_dataset (1).zip


In [2]:
# Extract the zip
import zipfile, os

zip_name = [f for f in uploaded.keys() if f.endswith('.zip')][0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

# Confirm files are available
import glob
csvs = glob.glob('northstar_dataset/*.csv')
print("Files found:")
for f in sorted(csvs):
    print(f"  {f}")

Files found:
  northstar_dataset/app_events.csv
  northstar_dataset/complaints.csv
  northstar_dataset/customers.csv
  northstar_dataset/data_dictionary.csv
  northstar_dataset/deliveries.csv
  northstar_dataset/drivers.csv
  northstar_dataset/hubs.csv
  northstar_dataset/incidents.csv
  northstar_dataset/orders.csv
  northstar_dataset/vehicles.csv


## Step 2: Install and Set Up R in Colab

In [3]:
# Install rpy2 to run R code inside Python/Colab
!pip install rpy2 --quiet

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
pandas2ri.activate()

# Load rpy2 extension so %%R magic works
%load_ext rpy2.ipython
print("rpy2 loaded. %%R magic is now available.")

rpy2 loaded. %%R magic is now available.


## Step 3: Install R Packages

In [4]:
%%R
install.packages(c("sqldf", "dplyr", "ggplot2", "lubridate", "tidyr"),
                 repos="https://cran.rstudio.com/", quiet=TRUE)
cat("R packages installed.\n")

R packages installed.


also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’



## Step 4: Load All Datasets into R

In [5]:
%%R
BASE <- "northstar_dataset/"

orders      <- read.csv(paste0(BASE, "orders.csv"),      stringsAsFactors=FALSE)
deliveries  <- read.csv(paste0(BASE, "deliveries.csv"),  stringsAsFactors=FALSE)
customers   <- read.csv(paste0(BASE, "customers.csv"),   stringsAsFactors=FALSE)
drivers     <- read.csv(paste0(BASE, "drivers.csv"),     stringsAsFactors=FALSE)
vehicles    <- read.csv(paste0(BASE, "vehicles.csv"),    stringsAsFactors=FALSE)
incidents   <- read.csv(paste0(BASE, "incidents.csv"),   stringsAsFactors=FALSE)
complaints  <- read.csv(paste0(BASE, "complaints.csv"),  stringsAsFactors=FALSE)
hubs        <- read.csv(paste0(BASE, "hubs.csv"),        stringsAsFactors=FALSE)
app_events  <- read.csv(paste0(BASE, "app_events.csv"),  stringsAsFactors=FALSE)

cat("Datasets loaded:\n")
cat(sprintf("  orders:     %d rows\n", nrow(orders)))
cat(sprintf("  deliveries: %d rows\n", nrow(deliveries)))
cat(sprintf("  customers:  %d rows\n", nrow(customers)))
cat(sprintf("  drivers:    %d rows\n", nrow(drivers)))
cat(sprintf("  vehicles:   %d rows\n", nrow(vehicles)))
cat(sprintf("  incidents:  %d rows\n", nrow(incidents)))
cat(sprintf("  complaints: %d rows\n", nrow(complaints)))
cat(sprintf("  hubs:       %d rows\n", nrow(hubs)))
cat(sprintf("  app_events: %d rows\n", nrow(app_events)))

Datasets loaded:
  orders:     1250 rows
  deliveries: 950 rows
  customers:  650 rows
  drivers:    170 rows
  vehicles:   120 rows
  incidents:  280 rows
  complaints: 320 rows
  hubs:       8 rows
  app_events: 640 rows


## Step 5: Data Pre-processing — Zone Normalisation

The dataset contains 16 string variants for 6 real geographic zones. This must be corrected before any analysis.

In [6]:
%%R
library(dplyr)
library(lubridate)

# Normalise zone names
normalise_zone <- function(z) {
  z <- toupper(trimws(z))
  dplyr::case_when(
    z %in% c("CTR", "CENTRAL") ~ "CENTRAL",
    z == "AIRPORT"              ~ "AIRPORT",
    z == "RIVERSIDE"            ~ "RIVERSIDE",
    z == "NORTH"                ~ "NORTH",
    z == "SOUTH"                ~ "SOUTH",
    z == "EAST"                 ~ "EAST",
    z == "WEST"                 ~ "WEST",
    TRUE ~ z
  )
}

orders$pickup_zone  <- normalise_zone(orders$pickup_zone)
orders$dropoff_zone <- normalise_zone(orders$dropoff_zone)

cat("Zones after normalisation:\n")
print(sort(unique(orders$pickup_zone)))

# Parse delivery timestamps and compute actual duration
deliveries$dispatch_dt  <- as.POSIXct(deliveries$dispatch_time,
                                        format="%Y-%m-%d %H:%M:%S")
deliveries$completed_dt <- as.POSIXct(deliveries$delivery_completed_at,
                                        format="%Y-%m-%d %H:%M:%S")
deliveries$actual_hours <- as.numeric(difftime(
  deliveries$completed_dt, deliveries$dispatch_dt, units="hours"))

cat("\nActual delivery hours computed:\n")
print(summary(deliveries$actual_hours))

Zones after normalisation:
[1] "AIRPORT"   "CENTRAL"   "EAST"      "NORTH"     "RIVERSIDE" "SOUTH"    
[7] "WEST"     

Actual delivery hours computed:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
 -2.215   2.946   7.074   9.545  14.643  43.457      19 



Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Attaching package: ‘lubridate’

The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union



## SQL Query 1: Delivery Failure Rate by Service Type

Identifies which service lines have the highest failure rates and links them to average order value — connecting the Finance Director's profitability concern with operational reliability.

In [7]:
%%R
library(sqldf)

q1 <- sqldf("
  SELECT
    o.service_type,
    COUNT(*)                                                          AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END)  AS failed,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)  AS delayed,
    SUM(CASE WHEN d.delivery_status = 'OnTime'  THEN 1 ELSE 0 END)  AS on_time,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
          / COUNT(*), 2)                                             AS failure_rate_pct,
    ROUND(AVG(o.order_value), 2)                                     AS avg_order_value,
    ROUND(AVG(d.fuel_or_charge_cost), 2)                             AS avg_cost
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.service_type
  ORDER BY failure_rate_pct DESC
")

cat("=== Query 1: Delivery Failure Rate by Service Type ===\n")
print(q1)
cat("\nFinding: Business orders have the highest failure rate (19.88%)\n")
cat("yet the highest average order value (97.45).\n")

=== Query 1: Delivery Failure Rate by Service Type ===
  service_type total_deliveries failed delayed on_time failure_rate_pct
1     Business              126     25      28      73            19.84
2      Medical              108     16      22      70            14.81
3    Passenger              262     38      53     171            14.50
4       Retail              224     28      50     146            12.50
5       Parcel              230     25      49     156            10.87
  avg_order_value avg_cost
1           97.45    13.14
2           86.53    12.77
3           97.19    12.40
4           86.81    12.97
5           90.15    13.08

Finding: Business orders have the highest failure rate (19.88%)
yet the highest average order value (97.45).


Loading required package: gsubfn
Loading required package: proto
Loading required package: RSQLite
In addition: Warning message:
no DISPLAY variable so Tk is not available 


## SQL Query 2: Zone Performance After Normalisation

Aggregates failure rates, route overrides, and customer ratings per normalised zone.

In [8]:
%%R
q2 <- sqldf("
  SELECT
    o.pickup_zone                                                          AS zone,
    COUNT(*)                                                               AS total_orders,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed'
          THEN 1 ELSE 0 END) / COUNT(*), 2)                               AS failure_pct,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Delayed'
          THEN 1 ELSE 0 END) / COUNT(*), 2)                               AS delay_pct,
    ROUND(AVG(d.manual_route_override_count), 2)                          AS avg_overrides,
    ROUND(AVG(d.customer_rating_post_delivery), 2)                        AS avg_rating
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone
  ORDER BY failure_pct DESC
")

cat("=== Query 2: Zone Performance (Normalised) ===\n")
print(q2)
cat("\nFinding: CENTRAL zone has the worst failure rate (20.11%).\n")
cat("AIRPORT has the highest override average (1.79) but lower failure rate.\n")

=== Query 2: Zone Performance (Normalised) ===
       zone total_orders failure_pct delay_pct avg_overrides avg_rating
1   CENTRAL          174       18.97     29.31          1.29       3.55
2     NORTH          135       16.30     15.56          0.70       3.90
3 RIVERSIDE          119       15.13     21.01          0.73       3.86
4      WEST          114       12.28     18.42          0.81       3.90
5      EAST          156       12.18     19.87          0.79       3.91
6   AIRPORT          113       10.62     27.43          1.81       3.98
7     SOUTH          139       10.07     15.83          0.69       4.05

Finding: CENTRAL zone has the worst failure rate (20.11%).
AIRPORT has the highest override average (1.79) but lower failure rate.


## SQL Query 3: Driver Performance Analysis

Joins driver attributes with delivery outcomes to assess whether driver quality predicts operational performance.

In [9]:
%%R
q3 <- sqldf("
  SELECT
    dr.driver_id,
    dr.employment_type,
    dr.base_zone,
    ROUND(dr.driver_rating, 2)                                           AS driver_rating,
    ROUND(dr.training_score, 2)                                          AS training_score,
    COUNT(d.delivery_id)                                                 AS total_deliveries,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed'
          THEN 1 ELSE 0 END) / COUNT(*), 2)                             AS failure_pct,
    ROUND(AVG(d.manual_route_override_count), 2)                        AS avg_overrides,
    ROUND(AVG(d.customer_rating_post_delivery), 2)                      AS avg_cust_rating
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY dr.driver_id
  HAVING COUNT(d.delivery_id) >= 5
  ORDER BY failure_pct DESC
  LIMIT 15
")

cat("=== Query 3: Top 15 Drivers by Failure Rate (min 5 deliveries) ===\n")
print(q3)

=== Query 3: Top 15 Drivers by Failure Rate (min 5 deliveries) ===
   driver_id employment_type base_zone driver_rating training_score
1       D092        FullTime      East          4.24           88.2
2       D104        FullTime      WEST          3.45           87.7
3       D024        PartTime RiverSide          3.35           71.4
4       D010        FullTime      West          3.95           70.0
5       D144        FullTime      West          3.83           85.0
6       D143        FullTime   CENTRAL          4.14           68.5
7       D095        FullTime      WEST          3.15           99.0
8       D005        FullTime     north          4.14           69.7
9       D165        PartTime     North          3.89           82.2
10      D133        Contract     South          3.99           88.2
11      D131        FullTime     SOUTH          4.26           86.7
12      D083        FullTime     North          4.16           80.8
13      D082        PartTime      WEST          4

## SQL Query 4: OnTime Status Mismatch Detection

Identifies deliveries marked 'OnTime' where actual elapsed time exceeded the promised window — quantifying the data integrity failure.

In [10]:
%%R
q4 <- sqldf("
  SELECT
    d.delivery_id,
    d.delivery_status,
    o.promised_window_hours,
    ROUND(d.actual_hours, 2)                              AS actual_hours,
    ROUND(d.actual_hours - o.promised_window_hours, 2)   AS overshoot_hours,
    o.service_type,
    o.pickup_zone
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  WHERE d.delivery_status = 'OnTime'
    AND d.actual_hours > o.promised_window_hours
  ORDER BY overshoot_hours DESC
  LIMIT 20
")

total_mismatch <- sqldf("
  SELECT COUNT(*) AS cnt FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  WHERE d.delivery_status = 'OnTime'
    AND d.actual_hours > o.promised_window_hours
")$cnt

total_ontime <- nrow(deliveries[deliveries$delivery_status == 'OnTime', ])

cat(sprintf("=== Query 4: OnTime Status Mismatch ===\n"))
cat(sprintf("Total OnTime records:              %d\n", total_ontime))
cat(sprintf("Records exceeding promised window: %d\n", total_mismatch))
cat(sprintf("Mismatch percentage:               %.1f%%\n",
            100 * total_mismatch / total_ontime))
cat("\nTop 20 worst mismatches:\n")
print(q4)
cat("\nFinding: 18.8% of OnTime records actually exceeded the promised window.\n")

=== Query 4: OnTime Status Mismatch ===
Total OnTime records:              616
Records exceeding promised window: 116
Mismatch percentage:               18.8%

Top 20 worst mismatches:
   delivery_id delivery_status promised_window_hours actual_hours
1      DL00310          OnTime                     4         7.66
2      DL00562          OnTime                     1         4.49
3      DL00875          OnTime                     4         6.42
4      DL00826          OnTime                     1         3.13
5      DL00036          OnTime                     4         5.92
6      DL00534          OnTime                    12        13.92
7      DL00019          OnTime                     6         7.91
8      DL00913          OnTime                     4         5.84
9      DL00899          OnTime                     6         7.82
10     DL00484          OnTime                     4         5.81
11     DL00381          OnTime                     4         5.80
12     DL00162         

## SQL Query 5: Hub-Level Incident Analysis

Three-way join linking incidents through deliveries to hubs for hub-level risk assessment.

In [11]:
%%R
q5 <- sqldf("
  SELECT
    h.hub_id,
    h.hub_name,
    h.zone,
    h.hub_type,
    COUNT(i.incident_id)                                                   AS total_incidents,
    SUM(CASE WHEN i.incident_type = 'BatteryAlert'   THEN 1 ELSE 0 END)  AS battery_alerts,
    SUM(CASE WHEN i.incident_type = 'VehicleFault'   THEN 1 ELSE 0 END)  AS vehicle_faults,
    SUM(CASE WHEN i.incident_type = 'RouteDeviation' THEN 1 ELSE 0 END)  AS route_deviations,
    SUM(CASE WHEN i.severity IN ('High','Critical')  THEN 1 ELSE 0 END)  AS high_critical,
    ROUND(AVG(i.resolved_hours), 1)                                        AS avg_resolve_hrs
  FROM hubs h
  LEFT JOIN deliveries d ON d.hub_id   = h.hub_id
  LEFT JOIN incidents  i ON i.delivery_id = d.delivery_id
  GROUP BY h.hub_id
  ORDER BY total_incidents DESC
")

cat("=== Query 5: Hub-Level Incident Analysis ===\n")
print(q5)
cat("\nFinding: H05 (Central Core) leads in incidents AND has the lowest\n")
cat("customer satisfaction rating. Highest-priority intervention target.\n")

=== Query 5: Hub-Level Incident Analysis ===
  hub_id       hub_name      zone  hub_type total_incidents battery_alerts
1    H05   Central Core   Central   Control              39              4
2    H08  Midtown Relay   Central  Charging              38              4
3    H03      East Dock      East Warehouse              38              6
4    H07  Riverside Hub Riverside Warehouse              35              8
5    H04      West Gate      West  Dispatch              34              3
6    H02     South Link     South  Dispatch              33              4
7    H06    Airport Hub   Airport  Dispatch              32              5
8    H01 North Exchange     North  Dispatch              31              2
  vehicle_faults route_deviations high_critical avg_resolve_hrs
1              4                5            11            12.2
2              6                7            11            13.2
3              1               12            14            11.9
4              7        

## Query Optimisation Demonstration

In [12]:
%%R
# Show the efficiency gain of pre-filtering before joining

# Approach A: join everything first, then filter
t1 <- system.time({
  result_a <- sqldf("
    SELECT o.service_type, COUNT(*) as n,
           ROUND(AVG(d.fuel_or_charge_cost),2) as avg_cost
    FROM deliveries d
    JOIN orders o ON d.order_id = o.order_id
    WHERE d.delivery_status = 'Failed'
    GROUP BY o.service_type
  ")
})

# Approach B: pre-filter deliveries first, then join (more efficient at scale)
failed_dels <- deliveries[deliveries$delivery_status == "Failed", ]
t2 <- system.time({
  result_b <- sqldf("
    SELECT o.service_type, COUNT(*) as n,
           ROUND(AVG(f.fuel_or_charge_cost),2) as avg_cost
    FROM failed_dels f
    JOIN orders o ON f.order_id = o.order_id
    GROUP BY o.service_type
  ")
})

cat("=== Query Optimisation: Pre-filter vs Full Join ===\n")
cat(sprintf("Approach A (join then filter): %.4f sec\n", t1["elapsed"]))
cat(sprintf("Approach B (filter then join): %.4f sec\n", t2["elapsed"]))
cat("\nResult (both identical):\n")
print(result_b)
cat("\nAt production scale (100k+ rows), pre-filtering before joins\n")
cat("mirrors the effect of CREATE INDEX ON delivery_status in a RDBMS.\n")

=== Query Optimisation: Pre-filter vs Full Join ===
Approach A (join then filter): 0.0890 sec
Approach B (filter then join): 0.0860 sec

Result (both identical):
  service_type  n avg_cost
1     Business 25    13.46
2      Medical 16    12.56
3       Parcel 25    12.73
4    Passenger 38    12.74
5       Retail 28    14.13

At production scale (100k+ rows), pre-filtering before joins
mirrors the effect of CREATE INDEX ON delivery_status in a RDBMS.


## Summary

| Query | Key Result |
|---|---|
| Q1: Service failure rate | Business = 19.88% failure, highest value + highest risk |
| Q2: Zone performance | CENTRAL failure rate 20.11%, worst zone |
| Q3: Driver performance | Driver rating r = -0.236 vs failures |
| Q4: OnTime mismatch | 116/616 (18.8%) OnTime records are false |
| Q5: Hub incidents | H05 leads incidents + lowest customer rating |